# Prueba de Diebold-Mariano: las diferencias del walk-forward, ¿son reales o ruido?

El walk-forward (`07`) ya mostro diferencias de MAE entre modelos en los 6 origenes, pero una
diferencia de MAE por si sola no dice si es estadisticamente significativa -- con series horarias
autocorrelacionadas, un t-test comun no es valido. Diebold-Mariano (1995) es la prueba estandar para
esto: compara la diferencia de perdida (aqui, error absoluto) entre dos modelos, con una varianza
robusta a autocorrelacion (HAC/Newey-West), y da un p-valor de si un modelo es significativamente
mejor que el otro.

Este notebook REPITE el entrenamiento de los 5 modelos en los 6 origenes (mismos hiperparametros
fijos que en `07` -- GARCH(1,1) normal, input_size=168h para N-BEATSx/N-HiTS, depth=3/lr=0.01 para
XGBoost), pero esta vez GUARDA las predicciones hora por hora en vez de solo las metricas agregadas,
porque `07` no las guardo y hacen falta para la prueba.

In [1]:
# --- Celda de arranque ---
import pandas as pd
import numpy as np
from pathlib import Path
import statsmodels.api as sm
import warnings
warnings.filterwarnings("ignore")

def encontrar_raiz_proyecto(marcador="requirements.txt"):
    actual = Path.cwd()
    for carpeta in [actual, *actual.parents]:
        if (carpeta / marcador).exists():
            return carpeta
    raise FileNotFoundError(f"No encontre '{marcador}' subiendo desde {actual}")

RAIZ = encontrar_raiz_proyecto()
print("Raiz del proyecto:", RAIZ)

def calcular_metricas(y_real, y_pred):
    y_real, y_pred = np.asarray(y_real, dtype=float), np.asarray(y_pred, dtype=float)
    error = y_real - y_pred
    mae = np.abs(error).mean()
    rmse = np.sqrt((error ** 2).mean())
    mape = (np.abs(error) / y_real).mean() * 100
    return mae, rmse, mape

def prueba_diebold_mariano(error_abs_1, error_abs_2, horizonte=24):
    """H0: los dos modelos tienen la misma precision. d = |e1| - |e2|.
    DM > 0 y p<0.05 => modelo 1 es SIGNIFICATIVAMENTE PEOR que modelo 2.
    DM < 0 y p<0.05 => modelo 1 es SIGNIFICATIVAMENTE MEJOR que modelo 2."""
    d = np.asarray(error_abs_1) - np.asarray(error_abs_2)
    X = np.ones_like(d)
    modelo_dm = sm.OLS(d, X).fit(cov_type="HAC", cov_kwds={"maxlags": horizonte - 1})
    return modelo_dm.tvalues[0], modelo_dm.pvalues[0]


Raiz del proyecto: C:\Users\mgdbj\xm-spot-price-predictor


In [2]:
# --- Cargar el dataset de features (mismo pipeline final que 07/08/09) ---
df_train = pd.read_csv(RAIZ / "data" / "processed" / "dataset_features_2019_2025.csv", parse_dates=["fecha_hora"])
df_test = pd.read_csv(RAIZ / "data" / "processed" / "dataset_features_2026.csv", parse_dates=["fecha_hora"])
df_completo = pd.concat([df_train, df_test], ignore_index=True).sort_values("fecha_hora").reset_index(drop=True)
print("Filas:", len(df_completo))


Filas: 65833


In [3]:
# --- Los mismos 6 origenes de 07 ---
origenes = [
    {"nombre": "Origen 1", "regimen": "La Nina (inicio)",       "corte_train": "2020-07-01", "test_inicio": "2020-07-01", "test_fin": "2020-09-30"},
    {"nombre": "Origen 2", "regimen": "La Nina (continuacion)", "corte_train": "2021-07-01", "test_inicio": "2021-07-01", "test_fin": "2021-09-30"},
    {"nombre": "Origen 3", "regimen": "La Nina (triple-dip)",   "corte_train": "2022-10-01", "test_inicio": "2022-10-01", "test_fin": "2022-12-31"},
    {"nombre": "Origen 4", "regimen": "El Nino (fuerte)",       "corte_train": "2023-10-01", "test_inicio": "2023-10-01", "test_fin": "2023-12-31"},
    {"nombre": "Origen 5", "regimen": "El Nino (pico)",         "corte_train": "2024-01-01", "test_inicio": "2024-01-01", "test_fin": "2024-03-31"},
    {"nombre": "Origen 6", "regimen": "El Nino 2026 (neutral->fuerte)", "corte_train": "2026-01-01", "test_inicio": "2026-01-01", "test_fin": "2026-08-05"},
]

columnas_excluir = ["fecha_hora", "precio_bolsa", "demanda", "generacion", "anio", "mes", "hora", "dia_semana", "dia_anio"]
columnas_features = [c for c in df_completo.columns if c not in columnas_excluir]
print(len(columnas_features), "features para XGBoost")


40 features para XGBoost


In [4]:
# --- Reentrenar los 5 modelos por origen, GUARDANDO predicciones hora por hora ---
import xgboost as xgb
from arch import arch_model

predicciones_crudas = []  # lista de dicts: origen, fecha_hora, real, modelo, prediccion

for o in origenes:
    train = df_completo[df_completo["fecha_hora"] < o["corte_train"]].dropna(subset=columnas_features).copy()
    test = df_completo[(df_completo["fecha_hora"] >= o["test_inicio"]) & (df_completo["fecha_hora"] <= o["test_fin"])].copy()

    # --- Persistencia ---
    for _, fila in test.iterrows():
        predicciones_crudas.append({"origen": o["nombre"], "fecha_hora": fila["fecha_hora"],
                                     "real": fila["precio_bolsa"], "modelo": "Persistencia",
                                     "prediccion": fila["precio_lag24h"]})

    # --- XGBoost ---
    X_train, y_train = train[columnas_features], np.log(train["precio_bolsa"])
    X_test = test[columnas_features]
    m_xgb = xgb.XGBRegressor(n_estimators=500, max_depth=3, learning_rate=0.01,
                              subsample=0.8, colsample_bytree=0.8, random_state=42)
    m_xgb.fit(X_train, y_train)
    pred_xgb = np.exp(m_xgb.predict(X_test))
    for fh, real, pred in zip(test["fecha_hora"], test["precio_bolsa"], pred_xgb):
        predicciones_crudas.append({"origen": o["nombre"], "fecha_hora": fh, "real": real,
                                     "modelo": "XGBoost", "prediccion": pred})

    # --- ARX+GARCH ---
    # Festivos agregados 2026-09-10: confirmado con significancia estadistica en N-BEATSx (p=0.0064).
    regresoras_arx = ["aportes_hidricos", "volumen_embalses", "oni", "es_pandemia",
                       "hora_sin", "hora_cos", "dia_semana_sin", "dia_semana_cos",
                       "es_festivo", "festivo_lag24h", "festivo_lag48h", "festivo_lag72h",
                       "festivo_lag168h", "mismatch_festivo_24h", "mismatch_festivo_168h"]
    train["precio_lag24h_log"] = np.log(train["precio_lag24h"])
    test["precio_lag24h_log"] = np.log(test["precio_lag24h"])
    columnas_x_arx = regresoras_arx + ["precio_lag24h_log"]

    y_train_log = np.log(train["precio_bolsa"])
    y_mean, y_std = y_train_log.mean(), y_train_log.std()
    y_train_arx = (y_train_log - y_mean) / y_std * 10
    X_train_raw = train[columnas_x_arx]
    x_mean, x_std = X_train_raw.mean(), X_train_raw.std()
    X_train_arx = (X_train_raw - x_mean) / x_std

    modelo_arx = arch_model(y_train_arx, x=X_train_arx, mean="ARX", lags=0, vol="GARCH", p=1, q=1, dist="normal")
    resultado_arx = modelo_arx.fit(disp="off", options={"maxiter": 500})

    X_test_arx = (test[columnas_x_arx] - x_mean) / x_std
    params_media = resultado_arx.params[["Const"] + columnas_x_arx]
    pred_escalado = params_media["Const"] + (X_test_arx * params_media[columnas_x_arx]).sum(axis=1)
    pred_arx = np.exp(pred_escalado / 10 * y_std + y_mean)
    for fh, real, pred in zip(test["fecha_hora"], test["precio_bolsa"], pred_arx.values):
        predicciones_crudas.append({"origen": o["nombre"], "fecha_hora": fh, "real": real,
                                     "modelo": "ARX+GARCH", "prediccion": pred})

    print(f"{o['nombre']} [{o['regimen']}]  Persistencia/XGBoost/ARX+GARCH listos")

df_pred_rapidos = pd.DataFrame(predicciones_crudas)
print("\nTotal filas (3 modelos rapidos):", len(df_pred_rapidos))


Origen 1 [La Nina (inicio)]  Persistencia/XGBoost/ARX+GARCH listos


Origen 2 [La Nina (continuacion)]  Persistencia/XGBoost/ARX+GARCH listos


C:\Users\mgdbj\AppData\Local\Temp\ipykernel_4568\2223122230.py:46: ConvergenceWarning: The optimizer returned code 8. The message is:
Positive directional derivative for linesearch
See scipy.optimize.fmin_slsqp for code meaning.

  resultado_arx = modelo_arx.fit(disp="off", options={"maxiter": 500})


Origen 3 [La Nina (triple-dip)]  Persistencia/XGBoost/ARX+GARCH listos


Origen 4 [El Nino (fuerte)]  Persistencia/XGBoost/ARX+GARCH listos


C:\Users\mgdbj\AppData\Local\Temp\ipykernel_4568\2223122230.py:46: ConvergenceWarning: The optimizer returned code 8. The message is:
Positive directional derivative for linesearch
See scipy.optimize.fmin_slsqp for code meaning.

  resultado_arx = modelo_arx.fit(disp="off", options={"maxiter": 500})


Origen 5 [El Nino (pico)]  Persistencia/XGBoost/ARX+GARCH listos


Origen 6 [El Nino 2026 (neutral->fuerte)]  Persistencia/XGBoost/ARX+GARCH listos

Total filas (3 modelos rapidos): 48258


In [5]:
# --- N-BEATSx y N-HiTS por origen, guardando predicciones crudas ---
from neuralforecast import NeuralForecast
from neuralforecast.models import NBEATSx, NHITS

hist_exog_dl = ["volumen_embalses", "aportes_hidricos", "demanda_lag24h"]
futr_exog_continua_dl = ["oni"]
# Festivos agregados 2026-09-10: confirmado con significancia estadistica en N-BEATSx (p=0.0064).
futr_exog_ya_escalada_dl = ["hora_sin", "hora_cos", "dia_semana_sin", "dia_semana_cos",
                             "es_festivo", "festivo_lag24h", "festivo_lag48h", "festivo_lag72h",
                             "festivo_lag168h", "mismatch_festivo_24h", "mismatch_festivo_168h"]
futr_exog_dl = futr_exog_continua_dl + futr_exog_ya_escalada_dl
INPUT_SIZE_DL = 168
MAX_STEPS_DL = 1000

predicciones_dl = []

for o in origenes:
    train = df_completo[df_completo["fecha_hora"] < o["corte_train"]].dropna(subset=columnas_features).copy()
    test = df_completo[(df_completo["fecha_hora"] >= o["test_inicio"]) & (df_completo["fecha_hora"] <= o["test_fin"])].copy()

    df_origen = pd.concat([train, test], ignore_index=True).sort_values("fecha_hora").reset_index(drop=True)
    mascara_train_o = df_origen["fecha_hora"] < o["corte_train"]

    df_nf_o = df_origen[["fecha_hora", "precio_bolsa"] + hist_exog_dl + futr_exog_dl].copy()
    for col in hist_exog_dl + futr_exog_continua_dl:
        mu, sigma = df_nf_o.loc[mascara_train_o, col].mean(), df_nf_o.loc[mascara_train_o, col].std()
        df_nf_o[col] = (df_nf_o[col] - mu) / sigma

    df_nf_o["unique_id"] = "precio_bolsa"
    df_nf_o = df_nf_o.rename(columns={"fecha_hora": "ds", "precio_bolsa": "y"})
    df_nf_o = df_nf_o[["unique_id", "ds", "y"] + hist_exog_dl + futr_exog_dl]

    n_test_o = int((~mascara_train_o).sum())
    n_windows_o = n_test_o // 24

    m_nbeatsx = NBEATSx(h=24, input_size=INPUT_SIZE_DL, hist_exog_list=hist_exog_dl, futr_exog_list=futr_exog_dl,
                          max_steps=MAX_STEPS_DL, val_check_steps=100, random_seed=42, enable_progress_bar=False)
    m_nhits = NHITS(h=24, input_size=INPUT_SIZE_DL, hist_exog_list=hist_exog_dl, futr_exog_list=futr_exog_dl,
                      max_steps=MAX_STEPS_DL, val_check_steps=100, random_seed=42, enable_progress_bar=False)
    nf_o = NeuralForecast(models=[m_nbeatsx, m_nhits], freq="h")
    cv_o = nf_o.cross_validation(df=df_nf_o, n_windows=n_windows_o, step_size=24)

    for col_modelo, nombre_modelo in [("NBEATSx", "N-BEATSx"), ("NHITS", "N-HiTS")]:
        for fh, real, pred in zip(cv_o["ds"], cv_o["y"], cv_o[col_modelo]):
            predicciones_dl.append({"origen": o["nombre"], "fecha_hora": fh, "real": real,
                                     "modelo": nombre_modelo, "prediccion": pred})

    print(f"{o['nombre']} [{o['regimen']}]  N-BEATSx/N-HiTS listos")

df_pred_dl = pd.DataFrame(predicciones_dl)
df_predicciones = pd.concat([df_pred_rapidos, df_pred_dl], ignore_index=True)
df_predicciones.to_csv(RAIZ / "data" / "processed" / "resultados" / "walkforward_predicciones_crudas.csv", index=False)
print("\nGuardado: walkforward_predicciones_crudas.csv --", len(df_predicciones), "filas")


2026-09-10 08:35:49,241	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


2026-09-10 08:35:49,458	INFO util.py:155 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


Seed set to 42


Seed set to 42


GPU available: False, used: False


...

`Trainer.fit` stopped: `max_steps=1000` reached.


GPU available: False, used: False


TPU available: False, using: 0 TPU cores


Origen 6 [El Nino 2026 (neutral->fuerte)]  N-BEATSx/N-HiTS listos



Guardado: walkforward_predicciones_crudas.csv -- 80418 filas


In [6]:
# --- Verificacion rapida: las metricas agregadas deberian coincidir con las de 07 ---
resumen_verificacion = df_predicciones.groupby(["origen", "modelo"]).apply(
    lambda g: pd.Series(calcular_metricas(g["real"], g["prediccion"]), index=["mae", "rmse", "mape"])
).reset_index()
resumen_verificacion.pivot(index="origen", columns="modelo", values="mae").round(2)


modelo,ARX+GARCH,N-BEATSx,N-HiTS,Persistencia,XGBoost
origen,,,,,
Origen 1,31.72,24.85,28.17,15.71,16.21
Origen 2,7.87,5.97,5.92,6.32,8.46
Origen 3,45.92,33.49,34.33,36.07,43.44
Origen 4,90.36,81.96,78.79,90.42,138.17
Origen 5,43.74,29.14,32.01,42.49,46.40
Origen 6,55.83,46.82,46.44,56.40,61.39


## Pruebas de Diebold-Mariano

Para cada origen, se compara cada par de modelos relevante. DM > 0 y p < 0.05 significa que el
primer modelo del par es SIGNIFICATIVAMENTE PEOR (pierde) contra el segundo. |DM| pequeno o p > 0.05
significa que la diferencia observada no es distinguible de ruido con esta muestra.

In [7]:
# --- DM test: N-BEATSx vs ARX+GARCH, N-BEATSx vs Persistencia, ARX+GARCH vs Persistencia, XGBoost vs Persistencia ---
pares_interes = [
    ("N-BEATSx", "ARX+GARCH"),
    ("N-BEATSx", "Persistencia"),
    ("N-HiTS", "ARX+GARCH"),
    ("N-HiTS", "Persistencia"),
    ("ARX+GARCH", "Persistencia"),
    ("XGBoost", "Persistencia"),
]

resultados_dm = []
for o in origenes:
    datos_origen = df_predicciones[df_predicciones["origen"] == o["nombre"]]
    for modelo_1, modelo_2 in pares_interes:
        d1 = datos_origen[datos_origen["modelo"] == modelo_1].sort_values("fecha_hora")
        d2 = datos_origen[datos_origen["modelo"] == modelo_2].sort_values("fecha_hora")
        # Alinear por fecha_hora, por si acaso
        merged = d1.merge(d2, on="fecha_hora", suffixes=("_1", "_2"))
        error_abs_1 = np.abs(merged["real_1"] - merged["prediccion_1"])
        error_abs_2 = np.abs(merged["real_2"] - merged["prediccion_2"])

        dm_stat, p_valor = prueba_diebold_mariano(error_abs_1, error_abs_2, horizonte=24)
        if p_valor < 0.05:
            conclusion = f"{modelo_1} PIERDE (sig.)" if dm_stat > 0 else f"{modelo_1} GANA (sig.)"
        else:
            conclusion = "sin diferencia significativa"

        resultados_dm.append({"origen": o["nombre"], "regimen": o["regimen"],
                               "modelo_1": modelo_1, "modelo_2": modelo_2,
                               "dm_stat": dm_stat, "p_valor": p_valor, "conclusion": conclusion})

df_dm = pd.DataFrame(resultados_dm)
df_dm.to_csv(RAIZ / "data" / "processed" / "resultados" / "diebold_mariano.csv", index=False)
print("Guardado: diebold_mariano.csv\n")
df_dm


Guardado: diebold_mariano.csv



,origen,regimen,modelo_1,modelo_2,dm_stat,p_valor,conclusion
0,Origen 1,La Nina (inicio),N-BEATSx,ARX+GARCH,-3.630962,2.823669e-04,N-BEATSx GANA (sig.)
1,Origen 1,La Nina (inicio),N-BEATSx,Persistencia,6.359460,2.024648e-10,N-BEATSx PIERDE (sig.)
2,Origen 1,La Nina (inicio),N-HiTS,ARX+GARCH,-1.567861,1.169135e-01,sin diferencia significativa
3,Origen 1,La Nina (inicio),N-HiTS,Persistencia,6.729384,1.703833e-11,N-HiTS PIERDE (sig.)
4,Origen 1,La Nina (inicio),ARX+GARCH,Persistencia,9.350822,8.696905e-21,ARX+GARCH PIERDE (sig.)
5,Origen 1,La Nina (inicio),XGBoost,Persistencia,0.661060,5.085738e-01,sin diferencia significativa
6,Origen 2,La Nina (continuacion),N-BEATSx,ARX+GARCH,-5.672714,1.405527e-08,N-BEATSx GANA (sig.)
7,Origen 2,La Nina (continuacion),N-BEATSx,Persistencia,-0.756077,4.496029e-01,sin diferencia significativa
8,Origen 2,La Nina (continuacion),N-HiTS,ARX+GARCH,-5.279068,1.298429e-07,N-HiTS GANA (sig.)
9,Origen 2,La Nina (continuacion),N-HiTS,Persistencia,-0.856673,3.916256e-01,sin diferencia significativa


In [8]:
# --- Resumen: en cuantos de los 6 origenes cada modelo le gana/pierde de forma SIGNIFICATIVA ---
for modelo_1, modelo_2 in pares_interes:
    subset = df_dm[(df_dm["modelo_1"] == modelo_1) & (df_dm["modelo_2"] == modelo_2)]
    gana_sig = (subset["conclusion"].str.contains("GANA")).sum()
    pierde_sig = (subset["conclusion"].str.contains("PIERDE")).sum()
    sin_dif = (subset["conclusion"] == "sin diferencia significativa").sum()
    print(f"{modelo_1:12s} vs {modelo_2:12s} -> gana sig.: {gana_sig}/6 | pierde sig.: {pierde_sig}/6 | sin diferencia: {sin_dif}/6")


N-BEATSx     vs ARX+GARCH    -> gana sig.: 6/6 | pierde sig.: 0/6 | sin diferencia: 0/6
N-BEATSx     vs Persistencia -> gana sig.: 3/6 | pierde sig.: 1/6 | sin diferencia: 2/6
N-HiTS       vs ARX+GARCH    -> gana sig.: 5/6 | pierde sig.: 0/6 | sin diferencia: 1/6
N-HiTS       vs Persistencia -> gana sig.: 3/6 | pierde sig.: 1/6 | sin diferencia: 2/6
ARX+GARCH    vs Persistencia -> gana sig.: 0/6 | pierde sig.: 3/6 | sin diferencia: 3/6
XGBoost      vs Persistencia -> gana sig.: 0/6 | pierde sig.: 5/6 | sin diferencia: 1/6
